# v1 PRA — Flat Rule Signal Investigation
# Market: player_points_rebounds_assists
# Hypothesis: is the under systematically mispriced? Segment by line tier, minutes, and outlier regression.

## Cell 1 — Imports & config

In [1]:
from pathlib import Path
import subprocess, sys
import numpy as np
import pandas as pd

repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
sys.path.insert(0, str(repo_root))
from src.nba_rebounds_modeling.duckdb_s3_creds import connect_duckdb_s3

market = 'player_points_rebounds_assists'

SEASONS = ["2023-24", "2024-25", "2025-26"]
SEASON_DATE_RANGES = {
    "2023-24": ("2023-10-01", "2024-06-30"),
    "2024-25": ("2024-10-01", "2025-06-30"),
    "2025-26": ("2025-10-01", "2026-06-30"),
}

def american_to_implied_prob(odds: float) -> float:
    if pd.isna(odds): return float("nan")
    if odds < 0: return (-odds) / ((-odds) + 100.0)
    return 100.0 / (odds + 100.0)

def american_profit(odds: float) -> float:
    if odds >= 0: return odds / 100.0
    return 100.0 / (-odds)

print(f"market: {market}")
print(f"MIN filter: >= 15")

market: player_points_rebounds_assists
MIN filter: >= 15


## Cell 2 — Load game logs + props, merge, compute PRA

In [2]:
con = connect_duckdb_s3()

# --- Game logs ---
logs_frames = []
for season in SEASONS:
    query = f"""
        SELECT
            PLAYER_NAME,
            CAST(PTS AS DOUBLE) AS PTS,
            CAST(REB AS DOUBLE) AS REB,
            CAST(AST AS DOUBLE) AS AST,
            CAST(MIN AS DOUBLE) AS MIN,
            GAME_DATE
        FROM read_csv_auto('s3://nba-api-mt/player_game_logs/{season}/*.csv',
                           header=true, ignore_errors=true)
    """
    frame = con.execute(query).df()
    frame["season"] = season
    logs_frames.append(frame)

logs = pd.concat(logs_frames, ignore_index=True)
logs = logs[logs["MIN"] > 0].copy()
logs["GAME_DATE"] = pd.to_datetime(logs["GAME_DATE"], format="mixed").dt.date
logs["player_key"] = logs["PLAYER_NAME"].str.lower().str.strip()

print(f"Game logs loaded: {len(logs):,} rows across {logs['season'].nunique()} seasons")
print(logs.groupby("season")["PLAYER_NAME"].count())

# --- Props ---
props_frames = []
for season in SEASONS:
    start_date, end_date = SEASON_DATE_RANGES[season]
    query = f"""
        SELECT
            player,
            CAST(prop_line AS DOUBLE) AS prop_line,
            CAST(over_odds AS DOUBLE) AS over_odds,
            CAST(under_odds AS DOUBLE) AS under_odds,
            game_time
        FROM read_csv_auto('s3://the-odds-api-mt/nba/historical_player_props/{season}/*.csv',
                           header=true, ignore_errors=true)
        WHERE market = '{market}'
          AND game_time >= '{start_date}'
          AND game_time <= '{end_date}'
    """
    frame = con.execute(query).df()
    frame["season"] = season
    props_frames.append(frame)

props_raw = pd.concat(props_frames, ignore_index=True)
props_raw["game_time"] = pd.to_datetime(props_raw["game_time"], format="mixed")
props_raw["game_date"] = props_raw["game_time"].dt.date
props_raw["player_key"] = props_raw["player"].str.lower().str.strip()

# Consensus: one row per (player_key, game_date, season) using median odds
props = (
    props_raw
    .groupby(["player_key", "game_date", "season"], as_index=False)
    .agg(
        player=("player", "first"),
        prop_line=("prop_line", "first"),
        over_odds=("over_odds", "median"),
        under_odds=("under_odds", "median"),
    )
)

print(f"\nProps loaded (consensus): {len(props):,} rows")
print(props.groupby("season")["player_key"].count())

# --- Merge ---
df = props.merge(
    logs[["player_key", "GAME_DATE", "PLAYER_NAME", "PTS", "REB", "AST", "MIN", "season"]],
    left_on=["player_key", "game_date"],
    right_on=["player_key", "GAME_DATE"],
    how="inner",
    suffixes=("", "_log"),
)

if "season_log" in df.columns:
    df["season"] = df["season"].fillna(df["season_log"])
    df = df.drop(columns=["season_log"], errors="ignore")

print(f"\nAfter join: {len(df):,} rows")
print(f"Props unmatched: {len(props) - len(df):,}")

# Compute PRA
df["PRA"] = df["PTS"] + df["REB"] + df["AST"]

# MIN >= 15 filter
before = len(df)
df = df[df["MIN"] >= 15].copy()
print(f"After MIN>=15 filter: {len(df):,} rows (dropped {before - len(df):,})")
print(df.groupby("season")["PLAYER_NAME"].count())

print("\nPRA distribution:")
print(df["PRA"].describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Game logs loaded: 81,034 rows across 3 seasons
season
2023-24    26399
2024-25    26303
2025-26    28332
Name: PLAYER_NAME, dtype: int64


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Props loaded (consensus): 41,081 rows
season
2023-24    10429
2024-25    13137
2025-26    17515
Name: player_key, dtype: int64

After join: 33,679 rows
Props unmatched: 7,402
After MIN>=15 filter: 32,305 rows (dropped 1,374)
season
2023-24     8132
2024-25    10566
2025-26    13607
Name: PLAYER_NAME, dtype: int64

PRA distribution:
count    32305.000000
mean        24.967033
std         11.412617
min          0.000000
25%         16.000000
50%         24.000000
75%         32.000000
max         95.000000
Name: PRA, dtype: float64


## Cell 3 — Line distribution audit

In [3]:
print("=== prop_line value counts ===")
print(df["prop_line"].value_counts().sort_index())

df["line_tier"] = pd.cut(df["prop_line"],
    bins=[0, 19.5, 24.5, 29.5, 200],
    labels=["low (<20)", "mid (20-25)", "high (25-30)", "star (30+)"])

print("\n=== line_tier counts ===")
print(df["line_tier"].value_counts())

=== prop_line value counts ===
prop_line
4.5        1
5.5        9
6.5       33
7.5       82
8.5      112
9.5      270
10.5     440
11.5     676
12.5     764
13.5     869
14.5    1138
15.5    1220
16.5    1347
17.5    1364
18.5    1431
19.5    1514
20.5    1456
21.5    1435
22.5    1400
23.5    1322
24.5    1295
25.5    1287
26.5    1181
27.5    1066
28.5     988
29.5     986
30.5     888
31.5     826
32.5     737
33.5     762
34.5     745
35.5     770
36.5     721
37.5     581
38.5     488
39.5     462
40.5     396
41.5     327
42.5     234
43.5     205
44.5     133
45.5      71
46.5      37
47.5      43
48.5      29
49.5      38
50.5      31
51.5      28
52.5      28
53.5      18
54.5       8
55.5       5
56.5       3
57.5       2
58.5       2
59.5       1
Name: count, dtype: int64

=== line_tier counts ===
line_tier
low (<20)       11270
star (30+)       8619
mid (20-25)      6908
high (25-30)     5508
Name: count, dtype: int64


## Cell 4 — Odds audit (vig filter)

In [4]:
df["vig"] = (
    df["over_odds"].apply(american_to_implied_prob)
    + df["under_odds"].apply(american_to_implied_prob)
)

print("=== Odds distribution ===")
print("over_odds:")
print(df["over_odds"].describe())
print("\nunder_odds:")
print(df["under_odds"].describe())
print(f"\nVig range: {df['vig'].min():.3f} – {df['vig'].max():.3f}")

df_clean = df[
    (df["vig"] >= 1.00) & (df["vig"] <= 1.20) &
    (df["over_odds"].abs() >= 5) &
    (df["under_odds"].abs() >= 5)
].copy()

n_total = len(df)
n_clean = len(df_clean)
print(f"\nRows passing vig filter: {n_clean:,} / {n_total:,} ({n_clean/n_total*100:.1f}%)")
print(f"Dropped: {n_total - n_clean:,}")
print(f"  - vig out of [1.00, 1.20]: {((df['vig'] < 1.00) | (df['vig'] > 1.20)).sum():,}")
print(f"  - near-zero odds: {((df['over_odds'].abs() < 5) | (df['under_odds'].abs() < 5)).sum():,}")
print(f"\nClean rows by season:")
print(df_clean.groupby("season")["PLAYER_NAME"].count())

=== Odds distribution ===
over_odds:
count    32305.000000
mean      -109.426405
std         31.198284
min       -155.000000
25%       -119.500000
50%       -115.000000
75%       -110.000000
max        152.000000
Name: over_odds, dtype: float64

under_odds:
count    32305.000000
mean      -112.420353
std         24.694889
min       -200.000000
25%       -120.000000
50%       -115.000000
75%       -110.000000
max        117.500000
Name: under_odds, dtype: float64

Vig range: 0.257 – 1.579

Rows passing vig filter: 31,278 / 32,305 (96.8%)
Dropped: 1,027
  - vig out of [1.00, 1.20]: 1,027
  - near-zero odds: 562

Clean rows by season:
season
2023-24     8018
2024-25    10276
2025-26    12984
Name: PLAYER_NAME, dtype: int64


## Cell 5 — Flat rule: bet every under

In [5]:
df_clean["y_under"] = (df_clean["PRA"] < df_clean["prop_line"]).astype(int)
df_clean["pnl"] = df_clean.apply(
    lambda r: american_profit(r["under_odds"]) if r["y_under"] == 1 else -1.0, axis=1
)

print(f"n={len(df_clean):,}  hit_rate={df_clean['y_under'].mean():.3f}  ROI={df_clean['pnl'].mean()*100:.2f}%")
print(df_clean.groupby("season")[["y_under","pnl"]].mean().rename(
    columns={"y_under":"hit_rate","pnl":"roi"}).round(3))

n=31,278  hit_rate=0.495  ROI=-7.49%
         hit_rate    roi
season                  
2023-24     0.507 -0.052
2024-25     0.488 -0.090
2025-26     0.493 -0.077


## Cell 6 — Segment by line tier

In [6]:
print("=== Under ROI by line tier ===")
print(df_clean.groupby("line_tier", observed=True).apply(lambda g: pd.Series({
    "n": len(g),
    "hit_rate": g["y_under"].mean(),
    "roi": g["pnl"].mean(),
}), include_groups=False).round(3))

print("\n=== Under ROI by season x tier ===")
print(df_clean.groupby(["season","line_tier"], observed=True)[["y_under","pnl"]].mean().round(3))

=== Under ROI by line tier ===
                    n  hit_rate    roi
line_tier                             
low (<20)     10421.0     0.467 -0.128
mid (20-25)    6799.0     0.503 -0.060
high (25-30)   5475.0     0.499 -0.067
star (30+)     8583.0     0.520 -0.027

=== Under ROI by season x tier ===
                      y_under    pnl
season  line_tier                   
2023-24 low (<20)       0.486 -0.091
        mid (20-25)     0.496 -0.073
        high (25-30)    0.491 -0.083
        star (30+)      0.534 -0.001
2024-25 low (<20)       0.463 -0.137
        mid (20-25)     0.508 -0.053
        high (25-30)    0.476 -0.111
        star (30+)      0.512 -0.045
2025-26 low (<20)       0.463 -0.135
        mid (20-25)     0.503 -0.058
        high (25-30)    0.523 -0.020
        star (30+)      0.514 -0.038


## Cell 7 — Calibration by line tier

In [7]:
df_clean["p_mkt_dv"] = (df_clean["over_odds"].apply(american_to_implied_prob)
                       / df_clean["vig"])
df_clean["y_over"] = (df_clean["PRA"] >= df_clean["prop_line"]).astype(int)

cal = df_clean.groupby("line_tier", observed=True).apply(lambda g: pd.Series({
    "mkt_p_over": g["p_mkt_dv"].mean(),
    "actual_over": g["y_over"].mean(),
    "gap": g["p_mkt_dv"].mean() - g["y_over"].mean(),
    "n": len(g),
}), include_groups=False)
print("=== Calibration gap by line tier (gap = mkt_p_over - actual_over_rate) ===")
print(cal.round(4))

=== Calibration gap by line tier (gap = mkt_p_over - actual_over_rate) ===
              mkt_p_over  actual_over     gap        n
line_tier                                             
low (<20)         0.4991       0.5330 -0.0339  10421.0
mid (20-25)       0.4995       0.4973  0.0022   6799.0
high (25-30)      0.4999       0.5010 -0.0012   5475.0
star (30+)        0.4996       0.4798  0.0198   8583.0


## Cell 8 — Minutes segmentation

In [8]:
df_clean["min_bucket"] = pd.cut(df_clean["MIN"], bins=[15, 25, 33, 38, 100],
                                 labels=["15-25","25-33","33-38","38+"])
print("=== Under ROI by line_tier x min_bucket ===")
print(df_clean.groupby(["line_tier","min_bucket"], observed=True)[["y_under","pnl"]].agg(
    count=("pnl","count"), hit_rate=("y_under","mean"), roi=("pnl","mean")
).round(3))

=== Under ROI by line_tier x min_bucket ===
                         count  hit_rate    roi
line_tier    min_bucket                        
low (<20)    15-25        4773     0.617  0.151
             25-33        4191     0.382 -0.287
             33-38        1204     0.238 -0.554
             38+           249     0.137 -0.746
mid (20-25)  15-25        1405     0.802  0.498
             25-33        3285     0.508 -0.051
             33-38        1592     0.319 -0.403
             38+           517     0.221 -0.586
high (25-30) 15-25         580     0.862  0.612
             25-33        2317     0.582  0.089
             33-38        1885     0.385 -0.281
             38+           693     0.228 -0.573
star (30+)   15-25         348     0.894  0.671
             25-33        2700     0.677  0.266
             33-38        3717     0.475 -0.112
             38+          1818     0.308 -0.424


## Cell 9 — Regression after outlier games hypothesis

In [9]:
dc = df_clean.sort_values(["PLAYER_NAME","season","game_date"]).copy()
dc["pra_last1"] = dc.groupby(["PLAYER_NAME","season"])["PRA"].shift(1)
dc["pra_roll10"] = dc.groupby(["PLAYER_NAME","season"])["PRA"].transform(
    lambda s: s.shift(1).rolling(10, min_periods=5).mean()
)

dc["last_game_vs_avg"] = dc["pra_last1"] / dc["pra_roll10"].replace(0, float("nan"))
dc["outlier_bucket"] = pd.cut(dc["last_game_vs_avg"],
    bins=[0, 0.7, 0.9, 1.1, 1.3, 10],
    labels=["cold (<0.7x)", "below avg", "on avg", "above avg", "hot (>1.3x)"]
)

print("=== Under ROI by last-game PRA vs rolling average ===")
print(dc.dropna(subset=["outlier_bucket"]).groupby("outlier_bucket", observed=True)[["y_under","pnl"]].agg(
    count=("pnl","count"), hit_rate=("y_under","mean"), roi=("pnl","mean")
).round(3))

=== Under ROI by last-game PRA vs rolling average ===
                count  hit_rate    roi
outlier_bucket                        
cold (<0.7x)     4461     0.485 -0.094
below avg        5842     0.496 -0.073
on avg           6956     0.512 -0.043
above avg        5128     0.503 -0.059
hot (>1.3x)      4550     0.482 -0.099


## Cell 10 — Pre-game proxy (rolling avg vs line)

In [10]:
dc["pra_roll10_vs_line"] = dc["pra_roll10"] - dc["prop_line"]

print("=== Under ROI by (rolling PRA - prop_line) ===")
dc["avg_vs_line"] = pd.cut(dc["pra_roll10_vs_line"],
    bins=[-50, -5, -2, 0, 2, 5, 50],
    labels=["avg << line", "avg < line", "avg ~line-", "avg ~line+", "avg > line", "avg >> line"]
)
print(dc.dropna(subset=["avg_vs_line"]).groupby("avg_vs_line", observed=True)[["y_under","pnl"]].agg(
    count=("pnl","count"), hit_rate=("y_under","mean"), roi=("pnl","mean")
).round(3))

=== Under ROI by (rolling PRA - prop_line) ===
             count  hit_rate    roi
avg_vs_line                        
avg << line    980     0.523 -0.025
avg < line    4568     0.504 -0.061
avg ~line-    7053     0.507 -0.053
avg ~line+    7498     0.499 -0.067
avg > line    5617     0.485 -0.092
avg >> line   1223     0.444 -0.166
